## Actividad Práctica Completa. Parte 3

**Materia:** Procesamiento de datos · **Tecnicatura:** Data Science and AI
**Autora:** Taisiia Bogatyreva · **Fecha:** 14.10.2026

## Dataset Adult

### Paso 1: Exploración inicial y diagnóstico de calidad	


In [1]:
import pandas as pd
import numpy as np

# Путь к файлу
RUTA = 'data/adult/adult.data'

columnas = ['age', 'workclass', 'fnlwgt', 'education', 'education-num',
            'marital-status', 'occupation', 'relationship', 'race', 'sex',
            'capital-gain', 'capital-loss', 'hours-per-week',
            'native-country', 'income']

# El archivo no tiene encabezado
df_raw = pd.read_csv(RUTA, header=None, names=columnas)


In [2]:
# ---------- 1. Estructura ----------
print(df_raw.shape)
print(df_raw.head())
print(df_raw.info())
print(df_raw.describe())

(32561, 15)
   age          workclass  fnlwgt   education  education-num  \
0   39          State-gov   77516   Bachelors             13   
1   50   Self-emp-not-inc   83311   Bachelors             13   
2   38            Private  215646     HS-grad              9   
3   53            Private  234721        11th              7   
4   28            Private  338409   Bachelors             13   

        marital-status          occupation    relationship    race      sex  \
0        Never-married        Adm-clerical   Not-in-family   White     Male   
1   Married-civ-spouse     Exec-managerial         Husband   White     Male   
2             Divorced   Handlers-cleaners   Not-in-family   White     Male   
3   Married-civ-spouse   Handlers-cleaners         Husband   Black     Male   
4   Married-civ-spouse      Prof-specialty            Wife   Black   Female   

   capital-gain  capital-loss  hours-per-week  native-country  income  
0          2174             0              40   United-S

In [3]:
# ---------- 2. Completitud ----------
print("\nNulos que ve pandas:")
print(df_raw.isnull().sum())

cols_texto = df_raw.select_dtypes(exclude='number').columns
print("\nValores '?' (nulos disfrazados):")
print(df_raw[cols_texto].apply(lambda s: (s.str.strip() == '?').sum()))


Nulos que ve pandas:
age               0
workclass         0
fnlwgt            0
education         0
education-num     0
marital-status    0
occupation        0
relationship      0
race              0
sex               0
capital-gain      0
capital-loss      0
hours-per-week    0
native-country    0
income            0
dtype: int64

Valores '?' (nulos disfrazados):
workclass         1836
education            0
marital-status       0
occupation        1843
relationship         0
race                 0
sex                  0
native-country     583
income               0
dtype: int64


In [4]:
# ---------- 3. Consistencia: espacios sobrantes ----------
print("\nValores con espacios al inicio/final:")
for col in cols_texto:
    n = (df_raw[col] != df_raw[col].str.strip()).sum()
    print(f"{col}: {n}")


Valores con espacios al inicio/final:
workclass: 32561
education: 32561
marital-status: 32561
occupation: 32561
relationship: 32561
race: 32561
sex: 32561
native-country: 32561
income: 32561


In [5]:

# ---------- 4. Unicidad ----------
print("\nFilas duplicadas:", df_raw.duplicated().sum())


Filas duplicadas: 24


In [6]:
# ---------- 5. Variable objetivo ----------
print("\nDistribución de income:")
print(df_raw['income'].value_counts(normalize=True))



Distribución de income:
income
<=50K    0.75919
>50K     0.24081
Name: proportion, dtype: float64


In [7]:
# ---------- 6. Cardinalidad y categorías raras ----------
print("\nValores únicos por columna de texto:")
print(df_raw[cols_texto].nunique())

frec = df_raw['native-country'].str.strip().value_counts(normalize=True)
print("\nPaíses con menos de 1% de las filas:")
print(frec[frec < 0.01])


Valores únicos por columna de texto:
workclass          9
education         16
marital-status     7
occupation        15
relationship       6
race               5
sex                2
native-country    42
income             2
dtype: int64

Países con menos de 1% de las filas:
native-country
Philippines                   0.006081
Germany                       0.004207
Canada                        0.003716
Puerto-Rico                   0.003501
El-Salvador                   0.003255
India                         0.003071
Cuba                          0.002918
England                       0.002764
Jamaica                       0.002488
South                         0.002457
China                         0.002303
Italy                         0.002242
Dominican-Republic            0.002150
Vietnam                       0.002058
Guatemala                     0.001966
Japan                         0.001904
Poland                        0.001843
Columbia                      0.001812
Taiwa

In [8]:

# ---------- 7. Numéricas: valores sospechosos ----------
print("\nMáximos de capital-gain y capital-loss:")
print(df_raw[['capital-gain', 'capital-loss']].max())
print("Filas con capital-gain == 99999:", (df_raw['capital-gain'] == 99999).sum())


Máximos de capital-gain y capital-loss:
capital-gain    99999
capital-loss     4356
dtype: int64
Filas con capital-gain == 99999: 159


Se cargó el archivo adult.data (32 561 filas y 15 columnas, sin encabezado). El diagnóstico mostró problemas en las cuatro dimensiones de calidad. Completitud: pandas no detecta nulos (isnull() devuelve 0), pero existen nulos disfrazados con el símbolo ?: 1836 en workclass, 1843 en occupation y 583 en native-country. Consistencia: los valores de las 9 columnas de texto tienen un espacio sobrante al inicio, y native-country presenta alta cardinalidad (42 valores, incluyendo ?) con muchas categorías de baja frecuencia; por ejemplo, Holand-Netherlands aparece una sola vez. Validez: capital-gain tiene 159 filas con el valor 99999, que parece un valor tope (centinela) y no un monto real. Unicidad: se encontraron 24 filas duplicadas. La variable objetivo income presenta un desbalanceo moderado: 75,9 % de <=50K y 24,1 % de >50K.

### Paso 2: Limpieza operativa	


In [9]:
df_clean = df_raw.copy()

cols_texto = df_clean.select_dtypes(exclude='number').columns

# 1. Quitar espacios sobrantes
for col in cols_texto:
    df_clean[col] = df_clean[col].str.strip()

# 2. '?' -> nulo real
df_clean = df_clean.replace('?', np.nan)

# 3. Duplicados
df_clean = df_clean.drop_duplicates().reset_index(drop=True)

print(df_clean.shape)
print(df_clean.isnull().sum())

(32537, 15)
age                  0
workclass         1836
fnlwgt               0
education            0
education-num        0
marital-status       0
occupation        1843
relationship         0
race                 0
sex                  0
capital-gain         0
capital-loss         0
hours-per-week       0
native-country     582
income               0
dtype: int64


In [10]:
print(df_clean[['workclass', 'occupation']].isnull().all(axis=1).sum())

1836


In [11]:
print(df_clean.groupby(df_clean['workclass'].isnull())[['age', 'hours-per-week']].mean())

                 age  hours-per-week
workclass                           
False      38.443536       40.949904
True       40.960240       31.919390


In [12]:
print(df_clean.groupby(df_clean['workclass'].isnull())['income'].value_counts(normalize=True))

workclass  income
False      <=50K     0.750888
           >50K      0.249112
True       <=50K     0.895969
           >50K      0.104031
Name: proportion, dtype: float64


Se eliminaron los espacios sobrantes de las 9 columnas de texto con .str.strip(), se reemplazó el símbolo ? por nulos reales (NaN) y se eliminaron 24 filas duplicadas, quedando 32 537 filas. Tras estas correcciones los nulos pasaron a ser visibles: 1836 en workclass, 1843 en occupation y 582 en native-country. Es importante el orden: primero se quitaron los espacios y luego se buscó ?, porque con el espacio inicial (' ?') el reemplazo no habría encontrado ninguna coincidencia. Cabe destacar que, al no existir un identificador de cliente, dos filas idénticas podrían corresponder a personas distintas, por lo que la eliminación de duplicados es una decisión que debe justificarse.


### Paso 3: Tratamiento de datos faltante

In [13]:
for col in ['workclass', 'occupation']:
    df_clean[col] = df_clean[col].fillna('Unknown')

print(df_clean.isnull().sum())
print(df_clean['workclass'].value_counts())

age                 0
workclass           0
fnlwgt              0
education           0
education-num       0
marital-status      0
occupation          0
relationship        0
race                0
sex                 0
capital-gain        0
capital-loss        0
hours-per-week      0
native-country    582
income              0
dtype: int64
workclass
Private             22673
Self-emp-not-inc     2540
Local-gov            2093
Unknown              1836
State-gov            1298
Self-emp-inc         1116
Federal-gov           960
Without-pay            14
Never-worked            7
Name: count, dtype: int64


In [14]:
print(df_clean[df_clean['workclass'] == 'Never-worked']['occupation'].value_counts())

occupation
Unknown    7
Name: count, dtype: int64


In [15]:
print(df_clean[df_clean['native-country'].isnull()]['workclass'].value_counts())

workclass
Private             409
Self-emp-not-inc     42
Self-emp-inc         42
Unknown              27
Local-gov            26
State-gov            19
Federal-gov          17
Name: count, dtype: int64


In [16]:
df_clean['native-country'] = df_clean['native-country'].fillna('Unknown')
print(df_clean.isnull().sum().sum())

0


Tras la limpieza aparecieron nulos reales: workclass (1836), occupation (1843) y native-country (582). Los nulos de workclass y occupation coinciden en las mismas filas; las 7 filas adicionales de occupation corresponden a workclass = Never-worked, es decir, nulos estructurales (quien nunca trabajó no tiene ocupación). Las personas sin workclass trabajan en promedio 31,9 horas semanales frente a 40,9, y solo 10,4 % gana más de 50 000 USD frente a 24,9 % en el resto: la ausencia depende de otras variables y se asocia al objetivo, por lo que no es MCAR y es compatible con MAR (sin una prueba estadística no puede confirmarse). Por ello no se eliminaron las filas y se imputó la categoría constante Unknown, que conserva la información de la ausencia y no requiere estadísticos calculados con los datos, evitando fuga de información. Los nulos de native-country, en cambio, no mostraron relación con workclass (27 de 582 con workclass nulo, 4,6 %, frente a 5,6 % en el total), por lo que parecen aleatorios; al ser pocos (1,8 %), también se imputó Unknown, por consistencia y para no calcular la moda con todo el conjunto antes de la partición. Tras la imputación no quedan valores nulos en el dataset.

### Paso 4: Categorías de baja frecuencia	


In [17]:
for col in df_clean.select_dtypes(exclude='number').columns:
    frec = df_clean[col].value_counts(normalize=True)
    print(col, '| категорий:', len(frec), '| редких (<1%):', (frec < 0.01).sum())

workclass | категорий: 9 | редких (<1%): 2
education | категорий: 16 | редких (<1%): 2
marital-status | категорий: 7 | редких (<1%): 1
occupation | категорий: 15 | редких (<1%): 2
relationship | категорий: 6 | редких (<1%): 0
race | категорий: 5 | редких (<1%): 2
sex | категорий: 2 | редких (<1%): 0
native-country | категорий: 42 | редких (<1%): 39
income | категорий: 2 | редких (<1%): 0


In [18]:
UMBRAL = 0.01

for col in ['workclass', 'native-country']:
    frec = df_clean[col].value_counts(normalize=True)
    print(col)
    print(frec[frec < UMBRAL].round(4))     # только редкие
    print()

workclass
workclass
Without-pay     0.0004
Never-worked    0.0002
Name: proportion, dtype: float64

native-country
native-country
Philippines                   0.0061
Germany                       0.0042
Canada                        0.0037
Puerto-Rico                   0.0035
El-Salvador                   0.0033
India                         0.0031
Cuba                          0.0029
England                       0.0028
Jamaica                       0.0025
South                         0.0025
China                         0.0023
Italy                         0.0022
Dominican-Republic            0.0022
Vietnam                       0.0021
Guatemala                     0.0019
Japan                         0.0019
Poland                        0.0018
Columbia                      0.0018
Taiwan                        0.0016
Haiti                         0.0014
Iran                          0.0013
Portugal                      0.0011
Nicaragua                     0.0010
Peru               

In [19]:
raras_wc = ['Without-pay', 'Never-worked']

print(df_clean[df_clean['workclass'].isin(raras_wc)]
      .groupby('workclass')['income'].value_counts())     # подсказка: raras_wc

print(df_clean.loc[df_clean['workclass'] == 'Never-worked', ['age', 'hours-per-week', 'occupation']])

workclass     income
Never-worked  <=50K      7
Without-pay   <=50K     14
Name: count, dtype: int64
       age  hours-per-week occupation
5359    18              40    Unknown
10842   23              35    Unknown
14767   17              30    Unknown
20328   18              10    Unknown
23217   20              40    Unknown
32281   30              40    Unknown
32291   18               4    Unknown


In [20]:
print(df_clean['native-country'].value_counts())
print(df_clean['workclass'].value_counts())

native-country
United-States                 29153
Mexico                          639
Unknown                         582
Philippines                     198
Germany                         137
Canada                          121
Puerto-Rico                     114
El-Salvador                     106
India                           100
Cuba                             95
England                          90
Jamaica                          81
South                            80
China                            75
Italy                            73
Dominican-Republic               70
Vietnam                          67
Guatemala                        62
Japan                            62
Poland                           60
Columbia                         59
Taiwan                           51
Haiti                            44
Iran                             43
Portugal                         37
Nicaragua                        34
Peru                             31
France       

In [21]:
print(df_clean.groupby('education')['education-num'].nunique().max())

1


In [22]:
print(df_clean[df_clean['workclass'].isin(['Without-pay', 'Never-worked'])]['income'].value_counts())

income
<=50K    21
Name: count, dtype: int64


In [23]:
print(df_clean[df_clean['workclass'] == 'Other']['income'].value_counts())

Series([], Name: count, dtype: int64)


Se analizó la frecuencia de las categorías de todas las columnas de texto con un umbral de 1 %. Tienen categorías por debajo del umbral: native-country (39 de 42), workclass (2 de 9: Without-pay, 14 filas, y Never-worked, 7 filas), education (2 de 16), occupation (2 de 15), marital-status (1 de 7) y race (2 de 5); relationship y sex no tienen. Las 21 filas de Without-pay y Never-worked pertenecen todas a la clase <=50K, y las 7 de Never-worked tienen occupation = Unknown (nulos estructurales, edades de 17 a 30 años), por lo que agruparlas no elimina información predictiva relevante. En este paso no se modificó ningún dato: el agrupamiento se realiza en el Paso 7 con OneHotEncoder(min_frequency=0,01), calculando las frecuencias solo con el conjunto de entrenamiento y sin fuga de datos. Se decidió no agrupar race (variable sensible, cuya agrupación ocultaría grupos minoritarios) y no usar education, que es redundante con education-num (a cada categoría le corresponde un único número).

In [24]:
print(df_clean['native-country'].nunique(), df_clean['workclass'].nunique())

42 9


### Paso 5: Valores atípicos y valores centinela	


In [25]:
print(df_clean['capital-gain'].sort_values().tail(200).unique())

[27828 34095 41310 99999]


Проверка скачка

In [26]:
df = df_clean.copy()     # подставь имя после Paso 4

vals = np.sort(df['capital-gain'].unique())   # уникальные значения (unique)
saltos = np.diff(vals)                      # разности соседних значений (diff)

print(vals[-5:])
print(saltos[-5:])
print("Max / предпоследнее:", vals[-1] / vals[-2])

[25236 27828 34095 41310 99999]
[  112  2592  6267  7215 58689]
Max / предпоследнее: 2.4206971677559914


La gente con los numeros 99999

In [27]:
mask = df['capital-gain'] == 99999

print("Строк с 99999:", mask.sum())               # sum
print(df.loc[mask, 'income'].value_counts())
print("Доля нулей:", (df['capital-gain'] == 0).mean().round(3))   # mean
print(df.loc[~mask & (df['capital-gain'] > 0), 'capital-gain'].describe())

Строк с 99999: 159
income
>50K    159
Name: count, dtype: int64
Доля нулей: 0.917
count     2553.000000
mean      7516.444575
std       5581.898066
min        114.000000
25%       3325.000000
50%       6514.000000
75%       8614.000000
max      41310.000000
Name: capital-gain, dtype: float64


IQR

In [28]:
num = df.select_dtypes(include='number').columns     # число

filas = []
for col in num:
    q1, q3 = df[col].quantile([0.25, 0.75])         # 0.25 и 0.75
    iqr = q3 - q1
    inf, sup = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_out = ((df[col] < inf) | (df[col] > sup)).sum()
    filas.append([col, q1, q3, inf, sup, n_out])

tabla = pd.DataFrame(filas, columns=['col', 'Q1', 'Q3', 'inf', 'sup', 'n_atipicos'])
print(tabla.round(1))

print(df[num].agg(['min', 'max']))

              col        Q1        Q3      inf       sup  n_atipicos
0             age      28.0      48.0     -2.0      78.0         142
1          fnlwgt  117827.0  236993.0 -60922.0  415742.0         993
2   education-num       9.0      12.0      4.5      16.5        1193
3    capital-gain       0.0       0.0      0.0       0.0        2712
4    capital-loss       0.0       0.0      0.0       0.0        1519
5  hours-per-week      40.0      45.0     32.5      52.5        9002
     age   fnlwgt  education-num  capital-gain  capital-loss  hours-per-week
min   17    12285              1             0             0               1
max   90  1484705             16         99999          4356              99


Se revisaron los valores atípicos y centinela de las columnas numéricas. En capital-gain, los valores más altos son 25 236, 27 828, 34 095, 41 310 y 99 999: la diferencia entre los dos últimos (58 689) es unas 8 veces mayor que la anterior (7 215) y no existe ningún valor intermedio. El valor 99 999 aparece en 159 filas y todas pertenecen a la clase >50K, por lo que no se trata de ruido sino de una señal fuerte; esto es coherente con la hipótesis de un tope o código especial, aunque los datos no permiten distinguir entre ambos. El 91,7 % de capital-gain son ceros y, sin el 99 999, el rango de los valores no nulos va de 114 a 41 310 (mediana 6 514). El método IQR no es útil en capital-gain ni capital-loss (Q1 = Q3 = 0, por lo que todos los valores no nulos, 2 712 y 1 519 filas, se marcan como atípicos), y en hours-per-week marca 9 002 filas (27,7 %) porque la mayoría trabaja exactamente 40 horas y el IQR es muy estrecho. Los atípicos de age (142, por encima de 78 años) y education-num (1 193, por debajo de 4,5) son valores reales y se conservan; fnlwgt es un peso muestral y no una característica de la persona. Decisión: no se eliminan filas ni se reemplaza 99 999 por nulos. En el Paso 7 se creará un indicador binario capital-gain == 99999 y se aplicará log1p a capital-gain y capital-loss, reglas fijas que no dependen de los datos y no generan fuga. La winsorización no se aplica, porque sus límites se calcularían con los datos y deberían ajustarse solo con train.

### Paso 6: Partición estratificada	


Разбиение

In [29]:
from sklearn.model_selection import train_test_split     # подсказка: train_test_split

df = df_clean.copy()      # результат Paso 4 (то же имя, что в Paso 5)

# 1) Целевая переменная в 0/1
y = (df['income'] == '>50K').astype(int)     # >50K -> 1

# 2) Признаки: всё, кроме income
X = df.drop(columns=['income'])              # drop

# 3) Разбиение
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,          # доля теста, обычно 0.2
    stratify=y,           # по какой переменной сохраняем доли?
    random_state=42        # любое число, например 42
)

print(X_train.shape, X_test.shape)

(26029, 14) (6508, 14)


In [30]:
print(df['income'].unique())

<StringArray>
['<=50K', '>50K']
Length: 2, dtype: str


In [31]:
print("Весь датасет:", y.mean().round(4))          # mean: доля класса 1
print("Train:       ", y_train.mean().round(4))
print("Test:        ", y_test.mean().round(4))

print(y_train.value_counts())
print(y_test.value_counts())

Весь датасет: 0.2409
Train:        0.2409
Test:         0.2409
income
0    19758
1     6271
Name: count, dtype: int64
income
0    4940
1    1568
Name: count, dtype: int64


In [32]:
# Индексы train и test не должны совпадать
print("Общих индексов:", len(set(X_train.index) & set(X_test.index)))

# Строки не потерялись
print(len(X_train) + len(X_test) == len(df))      # сравни с исходным df

Общих индексов: 0
True


Se dividió el dataset (32 537 filas, 14 características más la variable objetivo income) en entrenamiento y prueba con train_test_split, usando test_size = 0.2, stratify = y y random_state = 42. La variable objetivo se codificó como 1 para >50K y 0 para <=50K. El conjunto de entrenamiento tiene 26 029 filas (19 758 de clase <=50K y 6 271 de clase >50K) y el de prueba 6 508 (4 940 y 1 568). La proporción de >50K es 0,2409 en el dataset completo, en entrenamiento y en prueba, lo que confirma que la estratificación conservó el desbalance de clases. No hay índices comunes entre ambos conjuntos y no se perdió ninguna fila (26 029 + 6 508 = 32 537). La eliminación de duplicados se hizo antes de la partición (Paso 2), por lo que una misma fila no puede aparecer en ambos conjuntos. A partir de aquí, todo lo que se calcule a partir de los datos (codificación, escalado, límites o frecuencias) se ajustará solo con entrenamiento, dentro de un pipeline, y se aplicará después a prueba. Las frecuencias de categorías raras se calcularán solo con entrenamiento en el Paso 7

### Paso 7: Transformaciones (codificación y escalado)	

Preparación de columnas

In [33]:
X_train_p = X_train.copy()
X_test_p  = X_test.copy()

for d in (X_train_p, X_test_p):
    d['gain_99999'] = (d['capital-gain'] == 99999).astype(int)      # флаг
    # удаляем education и fnlwgt
    d.drop(columns=['education', 'fnlwgt'], inplace=True)

print(X_train_p.columns.tolist())

['age', 'workclass', 'education-num', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'gain_99999']


ColumnTransformer

In [34]:
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer

num_normal  = ['age', 'education-num', 'hours-per-week']
num_log     = ['capital-gain', 'capital-loss']
cat_cols    = X_train_p.select_dtypes(exclude='number').columns.tolist()
cat_con_min = [c for c in cat_cols if c != 'race']

log_pipe = Pipeline([
    ('log', FunctionTransformer(np.log1p, feature_names_out='one-to-one')),
    ('scale', StandardScaler()),
])

preproc = ColumnTransformer(
    transformers=[
        ('num',  StandardScaler(), num_normal),
        ('log',  log_pipe, num_log),
        ('cat',  OneHotEncoder(handle_unknown='ignore', min_frequency=0.01), cat_con_min),
        ('race', OneHotEncoder(handle_unknown='ignore'), ['race']),
    ],
    remainder='passthrough',
)

print(cat_con_min)

['workclass', 'marital-status', 'occupation', 'relationship', 'sex', 'native-country']


Fit en train, transform en test

In [35]:
X_train_t = preproc.fit_transform(X_train_p)     # fit_transform: учим параметры и применяем
X_test_t  = preproc.transform(X_test_p)      # transform: только применяем

print(X_train_t.shape, X_test_t.shape)

(26029, 52) (6508, 52)


Verificación sin fuga

In [36]:
nombres = preproc.get_feature_names_out()          # get_feature_names_out
print(len(nombres))
print(nombres[:10])

Xtr = X_train_t.toarray() if hasattr(X_train_t, 'toarray') else X_train_t
Xte = X_test_t.toarray()  if hasattr(X_test_t,  'toarray') else X_test_t

print("Train, первые 5 колонок:", Xtr[:, :5].mean(axis=0).round(3))
print("Test,  первые 5 колонок:", Xte[:, :5].mean(axis=0).round(3))

52
['num__age' 'num__education-num' 'num__hours-per-week' 'log__capital-gain'
 'log__capital-loss' 'cat__workclass_Federal-gov'
 'cat__workclass_Local-gov' 'cat__workclass_Private'
 'cat__workclass_Self-emp-inc' 'cat__workclass_Self-emp-not-inc']
Train, первые 5 колонок: [-0.  0. -0. -0.  0.]
Test,  первые 5 колонок: [ 0.013 -0.007  0.006 -0.001  0.009]


Categorías agrupadas por min_frequency

In [37]:
ohe = preproc.named_transformers_['cat']          # имя энкодера: 'cat'

for col, inf in zip(cat_con_min, ohe.infrequent_categories_):    # cat_con_min, не cat_cols
    print(col, '->', inf)

print([n for n in nombres if 'infrequent' in n])

workclass -> ['Never-worked' 'Without-pay']
marital-status -> ['Married-AF-spouse']
occupation -> ['Armed-Forces' 'Priv-house-serv']
relationship -> None
sex -> None
native-country -> ['Cambodia' 'Canada' 'China' 'Columbia' 'Cuba' 'Dominican-Republic'
 'Ecuador' 'El-Salvador' 'England' 'France' 'Germany' 'Greece' 'Guatemala'
 'Haiti' 'Holand-Netherlands' 'Honduras' 'Hong' 'Hungary' 'India' 'Iran'
 'Ireland' 'Italy' 'Jamaica' 'Japan' 'Laos' 'Nicaragua'
 'Outlying-US(Guam-USVI-etc)' 'Peru' 'Philippines' 'Poland' 'Portugal'
 'Puerto-Rico' 'Scotland' 'South' 'Taiwan' 'Thailand' 'Trinadad&Tobago'
 'Vietnam' 'Yugoslavia']
['cat__workclass_infrequent_sklearn', 'cat__marital-status_infrequent_sklearn', 'cat__occupation_infrequent_sklearn', 'cat__native-country_infrequent_sklearn']


Se aplicaron las transformaciones dentro de un ColumnTransformer, ajustado solo con el conjunto de entrenamiento (fit_transform en train, transform en prueba). Antes se eliminaron education (redundante con education-num) y fnlwgt (peso muestral, no una característica de la persona) y se creó el indicador gain_99999 (capital-gain == 99999), quedando 13 columnas. Las variables age, education-num y hours-per-week se estandarizaron con StandardScaler; capital-gain y capital-loss se transformaron con log1p y luego se estandarizaron (reglas fijas que no dependen de los datos). Las columnas de texto se codificaron con OneHotEncoder(handle_unknown='ignore'): para workclass, marital-status, occupation, relationship, sex y native-country se usó min_frequency = 0,01, de modo que las categorías con menos del 1 % del conjunto de entrenamiento se agrupan en una columna infrequent_sklearn (workclass: Never-worked y Without-pay; marital-status: Married-AF-spouse; occupation: Armed-Forces y Priv-house-serv; native-country: 39 de 42 países, quedando United-States, Mexico, Unknown e infrequent). race se codificó con un OneHotEncoder aparte, sin min_frequency, para no agrupar grupos minoritarios de una variable sensible. El resultado tiene 52 columnas en ambos conjuntos (26 029 y 6 508 filas). Como verificación, las medias de las primeras cinco columnas numéricas son 0 en entrenamiento y están entre -0,007 y 0,013 en prueba, lo que confirma que el escalado se ajustó solo con entrenamiento. Al hacerse el agrupamiento de categorías raras dentro del pipeline, la limitación de fuga del Paso 4 desaparece. Limitaciones: el umbral de 1 % es una elección arbitraria; native-country queda muy resumido; y todavía no se entrenó ningún modelo.